<a href="https://colab.research.google.com/github/frmariane-beep/reproducibility-of-llm-threat-modeling/blob/main/notebooks/threat_modeling_repro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import torch
print("GPU available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

GPU available: True
GPU name: Tesla T4


In [9]:
!pip install -q transformers accelerate bitsandbytes sentence-transformers langchain langchain-community keybert nltk faiss-cpu requests pandas huggingface_hub

In [10]:
import nltk
nltk.download('stopwords')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [11]:
from huggingface_hub import login
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

In [12]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

model_name = "meta-llama/Llama-2-7b-chat-hf"

# 4-bit quantization: shrinks the model so it fits comfortably on a free T4 GPU
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [13]:
prompt = "What is threat modeling, in one sentence?"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
output = model.generate(**inputs, max_new_tokens=60)
print(tokenizer.decode(output[0], skip_special_tokens=True))

[transformers] Both `max_new_tokens` (=60) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


What is threat modeling, in one sentence?
 Unterscheidung von Angriffspunkten und Angriffsverfahren in der IT-Sicherheit.

In English: Threat modeling is the process of identifying and categorizing potential attack points and attack methods in IT security.

Here are some key terms related
